In [ ]:
from pathlib import Path 
from subprocess import run
import os
import sys
import copykitten

# Authenticate w Github using SSH keys

- This notebook will walk you through the steps you need to follow configure your computer to use SSH to connect to github.

- The SSH protocol uses public key cryptography to authenticate with a server.

- You have a private key that is stored in your a folder on your computer. You will need to put a public key onto the server at github.com.

- As a first step, create an account on github.com. You'll supply an email address as a user id and a password. To secure your account, you should eventually set up a second factor for authentication.


SSH is a much more convenient way to connect with github. It is a particularly good way to sync a folder on your machine with a repo on github. 

Here are the steps to set up SSH authentication: 

1. Create a private key and a public key.

To do this, let's define a function. Then run this function with the name you want to use. For example, you might use "github" as the name when you run the function. 


In [ ]:
def start_ssh_agent():
    cmd = ['eval "$(ssh-agent -s)"']
    run(cmd, shell=True, capture_output=True)


In [ ]:
def ssh_add_key(key_name):
    """Runs one time after creating the new key pair."""
    s = str(Path.home() / (".ssh/" + key_name))
    if sys.platform == "darwin":
        cp = run(["ssh-add", s], capture_output=True)
        if cp.returncode:
            print(cp.stderr)
    elif sys.platform == "win32":
        s = "c:" + s
        cp = run(["ssh-add", s], capture_output=True)
        if cp.returncode:
            print(cp.stderr)


In [ ]:
def set_file_permissions(f: Path, mode: int):
    """Sets the permissions of a file.

    Args:
      f: The path for a file.
      mode: The permission mode.
    """
    try:
        os.chmod(f, mode)
    except OSError as e:
        print(f"Error setting permissions for {f}: {e}")

In [ ]:
def generate_keys(key_name):
    """Creates a key-pair (key_name, key_name.pub) and puts them in
    your .ssh folder.

    If you already have a keypair with these names, this will overwrite them.

    On macOS, the secret key is protected with a password.
    """
    home_path = Path.home().absolute()
    ssh_dir = home_path / ".ssh"
    if not ssh_dir.exists():
        ssh_dir.mkdir()
    key_path = ssh_dir / key_name

    if key_path.is_file():
        key_path.unlink()

    ls = [
        "ssh-keygen",
        "-q",
        "-t",
        "ed25519",
        "-f",
        str(key_path),
        "-N",
        "",
        "-C",
        key_name,
    ]
    cp = run(ls)
    if key_path.exists():
        set_file_permissions(key_path, 0o600)
    if key_path.with_suffix(".pub").exists():
        set_file_permissions(key_path.with_suffix(".pub"), 0o644)
    if cp.stderr:
        print(cp.stderr)
    return

In [ ]:
def name_config(url, key_name):
    ssh_dir = Path.home() / ".ssh"
    if not ssh_dir.exists():
        ssh_dir.mkdir()
    skey = (ssh_dir / key_name).exists()
    pkey = (ssh_dir / (key_name + ".pub")).exists()
    if skey or pkey:
        print()
        print("In your .ssh folder:")
        if skey:
            print(f"    You already have a secret key named {key_name}")
        if pkey:
            print(f"    You already have a public key named {key_name}" + ".pub")
        print()
        print("Delete the existing ones or try again with a new name.")
        return

    cfg = ssh_dir / "config"
    if not cfg.exists():
        cfg.touch()

    c = cfg.read_text()

    if url in c:
        existing_entry_with_url = True
        print()
        print("You seem to have an entry for this host in your config file.")
    else:
        existing_entry_with_url = False

    if key_name in c:
        print("")
        print("The existing config uses the keypair name you have entered. ")

    if url in c or key_name in c:
        print("Here is your existing config file")
        print()
        print(c)

    new_list = []
    new_list.append(f"Host {url}")
    new_list.append("  AddKeysToAgent yes")
    new_list.append("  IdentityFile " + str(Path.home()) + f"/.ssh/{key_name}")

    print("Here is the config block that will replace any existing one.")
    print()

    print("\n".join(new_list))
    print()

    lines = c.split("\n")
    if existing_entry_with_url:
        for n, s in enumerate(lines):
            if f"{url}" in s:
                n1 = n
                n2 = n1
                # short circuit eval prevents error in next line
                while not (n2 >= len(lines) or lines[n2] == ""):
                    n2 = n2 + 1
                break
        new_lines = lines
        new_lines[n1:n2] = new_list
        if not new_lines[-1] == "":
            new_lines.append("")
    elif not existing_entry_with_url:
        if lines == []:
            new_lines = new_list
        elif lines[-1] == "\n":
            new_lines = lines
            new_lines += new_list
        else:
            new_lines = lines
            new_lines.append("\n")
            new_lines += new_list

    cfg.write_text("\n".join(new_lines))
    generate_keys(key_name)
    start_ssh_agent()
    ssh_add_key(key_name)

    return


In [ ]:
key_name = "my_github_key7"

url = "github.com"

name_config(url, key_name)

In [ ]:
def pk_to_clipboard(key_name):
    """Use copykitten.copy() to put the public key on the clipboard"""
    p = Path.home() / (".ssh/" + key_name + ".pub")
    s = p.read_text()
    copykitten.copy(s)
    return


In [ ]:
pk_to_clipboard(key_name)